<a href="https://colab.research.google.com/github/marius-ne/CIE_ProjectB_Group13/blob/junchao/Program_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIE 2025/26 RWTH, PROJECT B, GROUP 13
Junchao Yu, Marius Neuhalfen

ToDo:
- Find defective nodes by comparing perfect structure and imperfect structures for all scenarios
- Group defective nodes into regions (arc sections or track sections)
- Predict whether structure is perfect or imperfect
- Predict where the imperfection lies

There are 25.XXX for deformation and 24.XXX for stress

# Setup

In [ ]:
# 1. 进入项目文件夹 (注意要用 %cd 而不是 !cd)
%cd CIE_ProjectB_Group13

# 2. 从远程拉取最新分支信息
!git fetch --all

# 3. 切换到名为 colab 的分支
!git checkout colab

# 4. (可选) 如果是第一次切换，可能需要追踪远程分支
# !git checkout -b colab origin/colab

In [ ]:
# --- 在所有 import 之前运行这段代码 ---
import sys
import importlib

# 如果系统里没有 imp (Python 3.12+)，我们手动造一个假的
if 'imp' not in sys.modules:
    sys.modules['imp'] = importlib
# ------------------------------------

# 然后再加载你的 autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
from _import import *

## Data getting (Colab only)


In [ ]:
import google.colab
google.colab.drive.mount("/content/drive")

Get ancillary data from Github

In [ ]:
!git clone https://github.com/marius-ne/CIE_ProjectB_Group13.git

## Load data (Colab only)

In [ ]:
os.getcwd()

**IMPORTANT:** You need to add a shortcut of the "programB" folder on GoogleDrive to your own "MyDrive" for this to work

In [ ]:
!ln -s /content/drive/MyDrive/programB data

# Go to data folder (Colab only)

In [ ]:
target_folder = "data/Data2"
current_folder = os.getcwd()

if Path(current_folder).name != target_folder:
    os.chdir(Path(current_folder) / Path(target_folder))
print(os.getcwd())


# Read data

In [ ]:
df = get_data_variable_aggregated((0,0,0,0), filter_out_invalid_nodes=False)
df[np.abs((df["TotalDeformation"] - np.sqrt(df["DirectionalDeformation_X_axis"]**2 + df["DirectionalDeformation_Y_axis"]**2 + df["DirectionalDeformation_Z_axis"]**2))) > 1e-3]

In [ ]:
df = get_data_variable_aggregated((0, 0, 0, 0))

In [ ]:
df = get_data_variable_and_region_aggregated([3,0,0,])


In [ ]:
df = get_data_all_aggregated(drop_invalid_nodes=True)

In [ ]:
res = get_delta_nodes(top_pct=2, drop_invalid_nodes=True)

In [ ]:
delta_nodes = np.unique(np.concatenate(list(res[0].values())))
# with open("delta_nodes_old_data_2_per_delta.pkl", "wb") as f:
#     pickle.dump(delta_nodes, f)

In [ ]:
# Print the largest absolute difference per variable for each scenario in diffs
for combo, diff_df in res[1].items():
    print(f"Combo: {combo}")
    # Exclude non-numeric columns
    # For each variable, print the max absolute difference and corresponding row
    for col in VARIABLE_NAMES:
        if col not in diff_df.columns:
            continue
        max_idx = diff_df[col].abs().idxmax()
        # idxmax returns a label (index), not a positional integer
        max_val = diff_df[col].loc[max_idx]
        row = diff_df.iloc[max_idx]
        print(f"  Variable: {col}")
        print(f"    Max abs diff: {max_val}")
        print(f"    Row: {row.to_dict()}")
    print("-" * 40)

In [ ]:
df

In [ ]:
df = get_data_variable_and_region_and_season_and_load_aggregated((3,))

In [ ]:
df = select_df_subset(df, [84,90])

In [ ]:
delta_nodes = get_delta_nodes_from_df(df, healthy_scenarios=[84], damaged_scenarios=[[90]], top_pct=0.001)
sorted(np.concatenate(list(delta_nodes[0].values())))

In [ ]:
NN = 1882
df[df["Node Number"]==NN & df["scenario"].isin([84,90])]
# Use parentheses to ensure correct operator precedence in boolean indexing
df_NN_84 = df[(df["Node Number"] == NN) & (df["scenario"] == 84)]
df_NN_90 = df[(df["Node Number"] == NN) & (df["scenario"] == 90)]

# Align on 'time' to subtract corresponding rows
df_NN_84 = df_NN_84.set_index("time").sort_index()
df_NN_90 = df_NN_90.set_index("time").sort_index()

# Subtract the two DataFrames (excluding non-numeric columns if needed)
diff = df_NN_84.select_dtypes(include="number") - df_NN_90.select_dtypes(include="number")
diff

In [ ]:
plot_nodes_time_series(df, 1882, variable="DirectionalDeformation_Z_axis")

In [ ]:
unequal = []
for node_number in df["Node Number"].unique():
    fft_90 = compute_node_fft(df, node_number=node_number, scenario_id=90, variable="DirectionalDeformation_X_axis")
    fft_84 = compute_node_fft(df, node_number=node_number, scenario_id=84, variable="DirectionalDeformation_X_axis")
    if not np.array_equal(fft_90, fft_84):
        print(node_number)
        print(f"Difference f: {np.abs(fft_90[0] - fft_84[0]).max()}")
        print(f"Difference m: {np.abs(fft_90[1] - fft_84[1]).max()}")
        unequal.append(node_number)

In [ ]:
compare_bridge_dynamics(df, 24356, 84, 90, variable="EquivalentStress")

In [ ]:
vx = create_multichannel_voxel_dataset(df)
vx[0].shape

In [ ]:
visualize_voxel_snapshot(vx[0])

In [ ]:
# df = get_data_all_aggregated()

In [ ]:
df.to_csv("data/Data2/df_all2.csv")

In [ ]:
df = pd.read_csv("data/Data2/df_all.csv")

# Data inspection

Compare stresses

In [ ]:
df_stress_healthy = read_data_file(0, 0, 0, 0, 4, filter_out_invalid_nodes=True)
df_stress_unhealthy = read_data_file(0, 0, 0, 6, 4, filter_out_invalid_nodes=True)
df_stress_healthy

## Obtain delta nodes

Note: there is a large difference between scenarios (3, 1, 1) (179 delta nodes) and (1, 1, 1) (0 delta nodes)!

In [ ]:
delta_nodes, diffs = get_delta_nodes(top_pct=0.0001)

In [ ]:
{k:len(v) for k, v in delta_nodes.items()}, len(set(np.concatenate(list(delta_nodes.values()))))

Get superset of delta nodes

In [ ]:
delta_nodes_superset = set()
for nodes in delta_nodes.values():
    delta_nodes_superset.update(nodes)

delta_nodes_superset = [int(x) for x in np.asarray(list(delta_nodes_superset)).tolist()]
# with open("delta_nodes_superset_0.0001'_top_pct.pkl", "wb") as f:
    # pickle.dump(delta_nodes_superset, f)

In [ ]:
df = get_data_variable_aggregated((0,0,0,0), filter_out_invalid_nodes=True)
plot_bridge_3d_variable_over_time_df(df, "TotalDeformation")


In [ ]:
combo = (2, 0, 1, 2, 3)
var_name = VARIABLE_NAMES[combo[-1]]
diff = get_variable_difference_between_combinations(
    (*combo[:3], 0, combo[-1]),
    combo,
    top_pct=0.01
)
plot_bridge_3d_variable_over_time_df(diff, var_name)

In [ ]:
diff

Compare deformations

In [ ]:
df_diff = get_variable_difference_between_combinations((0, 0, 0, 0, 0), (0, 0, 0, 1, 0), "TotalDeformation")

In [ ]:
df_diff

In [ ]:
plot_bridge_3d_variable_over_time_df(df_diff, "TotalDeformation")

# Visualize bridge structure

Show deformation only for changing nodes (i.e. )

In [ ]:
combo = (0, 1, 1, 6, 1)
var = VARIABLE_NAMES[combo[4]]
df_agg = get_data_variable_aggregated(combo[:-1])

In [ ]:
plot_bridge_3d_variable_over_time_df(
    df_agg, "TotalDeformation", title=combination_to_string(combo)
)

In [ ]:
plot_bridge_3d_structure(highlight_nodes=delta_nodes_superset)

# Training

Check scenarios

In [ ]:
df["scenario"].unique(), len(df["scenario"].unique())

Standardize

In [ ]:
variable_data = df.drop(columns=["Node Number", "time", "scenario", "health"])
variable_data_standardized = standardize(variable_data)

# Add indicators, node number and time back on
for col in ["Node Number", "time", "scenario", "health"]:
    variable_data_standardized[col] = df[col]

# columns = ["Node Number", "health", "time", "X", "Y", "Z"] + VARIABLE_NAMES
columns = ["Node Number", "time", "scenario", "health", "X", "Y", "Z"] + VARIABLE_NAMES
variable_data_standardized = variable_data_standardized[columns]
variable_data_standardized

## Dimensionality reduction

Select certain nodes

In [ ]:
# node_subset = set(NODES_MISSING_STRESS) - set(NODES_MISSING_DEFORMATION)

with open("delta_nodes_new_data_5_per_delta.pkl", "rb") as f:
    delta_nodes_superset = pickle.load(f)
node_subset = delta_nodes_superset

PCA

In [ ]:
X_pca, y, pca, scaler = apply_bridge_pca(
    variable_data_standardized, n_components=50
)


In [ ]:
X_pca["scenario"].unique()

Go with the node subset directly

In [ ]:

# Create y vector based on health status per scenario and time
health_by_scenario_time = df.groupby(['scenario', 'time'])['health'].first().unstack()
# Convert wide to long format for use alongside X_raw
health_by_scenario_time_long = health_by_scenario_time.stack().reset_index()
health_by_scenario_time_long.columns = ['scenario', 'time', 'health']

# Create X_raw
X_filtered = variable_data_standardized[variable_data_standardized['Node Number'].isin(node_subset)]

X_wide = reshape_multi_variable_to_wide(X_filtered)
# Align y to the same row order as X_wide after reshape_multi_variable_to_wide
# Assume X_wide has columns 'scenario' and 'time' that match health_by_scenario_time_long
# Align health_by_scenario_time_long to the same row order as X_wide
health_by_scenario_time_long = health_by_scenario_time_long.set_index(['scenario', 'time']).loc[
    X_wide.set_index(['scenario', 'time']).index
].reset_index()

y_raw = health_by_scenario_time_long['health']
y_binary = (y_raw > 0).astype(int)

X_wide

## Training

Remove ordinal data again (was used before just to seperate scenarios in the wide format)

In [ ]:
X_wide.drop(columns=["health","scenario"], inplace=True)
X_wide

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Get unique scenarios
unique_scenarios = health_by_scenario_time_long['scenario'].unique()

# Ensure equal distribution of health classes per scenario for stratification

# Create a DataFrame with one row per scenario, using the majority health label for each scenario
# Intentionally introduce data leakage: assign some scenarios to both train and test
scenario_health = health_by_scenario_time_long.groupby('scenario')['health'].agg(lambda x: x.value_counts().idxmax()).reset_index()

# Select a subset of scenarios to be in both train and test
leak_scenarios = scenario_health['scenario'].sample(frac=0.5, random_state=42).values

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=20)
train_idx, test_idx = next(sss.split(scenario_health['scenario'], scenario_health['health']))

scenarios_train = scenario_health['scenario'].iloc[train_idx].values
scenarios_test = scenario_health['scenario'].iloc[test_idx].values

# Add leak_scenarios to both train and test
scenarios_train = np.unique(np.concatenate([scenarios_train, leak_scenarios]))
scenarios_test = np.unique(np.concatenate([scenarios_test, leak_scenarios]))

# # Stratified split on scenario, using the majority health label as the stratification target
# sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=20)
# train_idx, test_idx = next(sss.split(scenario_health['scenario'], scenario_health['health']))

# scenarios_train = scenario_health['scenario'].iloc[train_idx].values
# scenarios_test = scenario_health['scenario'].iloc[test_idx].values

# Split scenarios into train/test
# scenarios_train, scenarios_test = train_test_split(
    # unique_scenarios, test_size=0.25, random_state=8
# )

# Create boolean masks for train/test based on scenario
train_mask = health_by_scenario_time_long['scenario'].isin(scenarios_train)
test_mask = health_by_scenario_time_long['scenario'].isin(scenarios_test)

# Split X_wide and y_binary accordingly
X_train = X_wide[train_mask].reset_index(drop=True)
X_test = X_wide[test_mask].reset_index(drop=True)
y_train = y_binary[train_mask].reset_index(drop=True)
y_test = y_binary[test_mask].reset_index(drop=True)

print("Train class distribution:\n", y_train.value_counts())
print("Test class distribution:\n", y_test.value_counts())
print(X_train.groupby(y_train).mean())

In [ ]:
X_train

Normal

In [ ]:
# X_raw, y_raw = df_train.drop(columns=["region","health"]), df_train["health"]
# X_train_raw, X_test_raw, y_train, y_test = train_test_split(
#     X_raw, y_raw, test_size = 0.1, random_state = 0, shuffle=True,
# )

In [ ]:


# X_train, X_test = standardize(X_train_raw, X_test_raw)

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = sklearn.ensemble.RandomForestClassifier(
    class_weight='balanced'
)
model.fit(X_train, y_train)

In [ ]:
print("Score: ", model.score(X_test, y_test))

Decision Tree

In [ ]:
# model = sklearn.tree.DecisionTreeClassifier()
# model.fit(X_train, y_train)

In [ ]:
# model.score(X_test, y_test), model.score(X_train, y_train)

Neural Network

In [ ]:
model = sklearn.neural_network.MLPClassifier(
    hidden_layer_sizes=(1000,1000,1000,1000),
    verbose=1,
)
model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test), model.score(X_train, y_train)

## Detailed report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Predict on test set
y_pred = model.predict(X_test)

# Print detailed classification report
print("Classification Report:")
report = classification_report(y_test, y_pred, output_dict=True)
print(classification_report(y_test, y_pred))

# Print confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# --- Plot Confusion Matrix ---
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion Matrix")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()

# --- Plot F1-score, Precision, Recall ---
metrics = ["precision", "recall", "f1-score"]
classes = [str(c) for c in sorted(np.unique(y_test))]
scores = {m: [report[c][m] for c in classes] for m in metrics}

plt.figure(figsize=(6, 4))
x = np.arange(len(classes))
width = 0.2
for i, m in enumerate(metrics):
    plt.bar(x + i*width, scores[m], width=width, label=m.capitalize())
plt.xticks(x + width, classes)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.xlabel("Class")
plt.title("Precision, Recall, F1-score per Class")
plt.legend()
plt.tight_layout()
plt.show()

# --- Plot individual predictions vs actual values ---
plt.figure(figsize=(12, 3))
plt.plot(y_test.values, 'o-', label="Actual", alpha=0.7)
plt.plot(y_pred, 'x-', label="Predicted", alpha=0.7)
plt.title("Individual Predictions vs Actual Values")
plt.xlabel("Sample Index")
plt.ylabel("Class")
plt.legend()
plt.tight_layout()
plt.show()